# 05 - Decision demo: confidence, routing and human-in-the-loop

Companion notebook to `python -m energymind.pipeline`. ~20 cases go through the
full loop: **seen before?** -> agents/reuse -> confidence `f` -> tier -> audit log,
then the calibration report, the risk-coverage curve and the human review cards.

Prerequisites: `python -m energymind.inject` and `python -m energymind.features`.

In [ ]:
import warnings

import pandas as pd

from energymind import io, pipeline
from energymind.config import load_settings

warnings.filterwarnings("ignore", category=FutureWarning)
settings = load_settings()

report = pipeline.run(settings, n_val=40, n_test=20)
print(
    f"val={report['n_val']} test={report['n_test']} | "
    f"test_error={report['test_error']} ECE={report['ece_test']}"
)
print("thresholds:", report["thresholds"])

## Confidence model `f`

`f = sigmoid(b0 + bA*logit(A) + bL*L + bH*logit(H) + bG*G + bQ*Q + bS*S)`, weights
learned on the validation split.

In [ ]:
pd.Series(report["coefficients"]).round(4).to_frame("beta")

## Routing (three tiers)

Tier 1 autonomous, Tier 2 human approval, Tier 3 human-led.

In [ ]:
cases = pd.DataFrame(report["cases"])
cases["split"].value_counts().to_frame("cases")

In [ ]:
test_cases = cases[cases.split == "test"].copy()
tier_names = {1: "1 autonomous", 2: "2 approval", 3: "3 human-led"}
test_cases["tier"] = test_cases["tier"].map(tier_names)
(
    test_cases[["anomaly_id", "f", "tier", "action", "origin", "true_origin", "converged", "correct"]]
    .round(4)
    .style.hide(axis="index")
)

## Human-in-the-loop: the review cards

Tier 2 (uncertain) cards ask the human to approve/reject the agents' proposal;
tier 3 (high-risk) cards ask the human for their own action and origin. Batch runs
use `--human-answer` (tier 2) and `--human-action/--human-origin` (tier 3).

In [ ]:
print(report["human_review"])
for case_id, card in list(report["review_cards"].items())[:1]:
    print(f"\n--- {case_id} ---")
    print(card)

In [ ]:
rc = report["risk_coverage"]
print(f"area under risk-coverage (lower is better):")
print(f"  f           : {rc['auc_risk_f']}")
print(f"  model_conf  : {rc['auc_risk_model_conf']}")

## Audit trail

Every deliberation message and every decision is a hash-chained `audit_event`.

In [ ]:
events = pd.DataFrame(report["audit_events"])
print("chain valid:", report["audit"]["valid"], "| events:", report["audit"]["events"])
events["event_type"].value_counts().to_frame("count")

In [ ]:
# one case's audit trail, start to finish
one = test_cases.iloc[0]["anomaly_id"]
events[events.anomaly_id == one].head(20)